In [0]:
from pyspark.sql import functions as F

In [0]:
def validate_transaction(df):
    check_df = (
    df
    .withColumn(
        'calculated_totalPrice',
        F.col('quantity') * F.col('unitPrice')
    )
    .withColumn(
        'dq_reason',
        F.when(F.col('transactionID').isNull(), F.lit('Missing_TransactionID'))
        .when(F.col('dateTime').isNull(), F.lit('Missing_DateTime'))
        .when(F.col('product').isNull() | (F.col('product').rlike('^[0-9]+$')), F.lit('Invalid_Product'))
        .when(F.col('quantity').isNull() | (F.col('quantity') <= 0), F.lit('Invalid_Quantity'))
        .when(F.col('unitPrice').isNull() | (F.col('unitPrice') <= 0), F.lit('Invalid_UnitPrice'))
        .when(F.col('totalPrice').isNull() | (F.col('totalPrice') <0), F.lit('Invalid_TotalPrice'))
        .when(F.col('totalPrice') != F.col('calculated_totalPrice'), F.lit('Invalid_TotalPrice_sum')).otherwise(F.lit(None))
    )
    .withColumn(
        'dq_status',
        F.when(F.col('dq_reason').isNull(), F.lit('VALID')).otherwise(F.lit('INVALID'))
    )
    .withColumn('checked_at', F.current_timestamp())
)
    return check_df

In [0]:
def split_valid_and_rejected(df):
    valid_df = df.filter(
        F.col('dq_status') == 'VALID'
    )
    invalid_df = df.filter(
        F.col('dq_status') == 'INVALID'
    )

    return valid_df, invalid_df
